# 🏛️ Termo Fácil — Backend Completo

**Sistema Inteligente de Redação de Termos de Depoimentos — SSP-PI**

Este notebook contém **todo o código-fonte do backend** do projeto Termo Fácil, organizado célula a célula conforme a estrutura real de arquivos do projeto.

---

## 📂 Estrutura do backend

```
backend/
├── requirements.txt
├── .env
├── app/
│   ├── main.py
│   ├── db.py
│   ├── models.py
│   ├── core/
│   │   ├── config.py
│   │   └── celery_app.py
│   ├── api/
│   │   ├── api.py
│   │   ├── deps.py
│   │   └── endpoints/
│   │       ├── admin.py
│   │       ├── auth.py
│   │       ├── jobs.py
│   │       ├── pdf.py
│   │       └── upload.py
│   ├── schemas/
│   │   ├── admin.py
│   │   └── job.py
│   ├── services/
│   │   ├── asr_service.py
│   │   ├── minio_service.py
│   │   └── pdf_service.py
│   └── tasks/
│       └── process_audio.py
└── scripts/
    └── seed_db.py
```

---

> ⚠️ **Pré-requisitos no Colab:**  
> Execute as células na ordem. A célula de instalação de dependências e a de configuração de variáveis de ambiente devem ser rodadas antes de qualquer outra.

---
## 1️⃣ Instalação de Dependências
`requirements.txt`

In [ ]:
# requirements.txt
# Instala todas as dependências do projeto
!pip install -q \
    asyncpg \
    boto3 \
    celery \
    fastapi \
    openai-whisper \
    pg8000 \
    pydantic \
    pydantic-settings \
    python-dotenv \
    python-multipart \
    redis \
    reportlab \
    sqlalchemy \
    uvicorn

print("✅ Dependências instaladas com sucesso!")

---
## 2️⃣ Criação da Estrutura de Pastas

Recria no Colab a mesma hierarquia de diretórios do projeto.

In [ ]:
import os

dirs = [
    "app/core",
    "app/api/endpoints",
    "app/schemas",
    "app/services",
    "app/tasks",
    "scripts",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# Cria arquivos __init__.py necessários
init_files = [
    "app/__init__.py",
    "app/core/__init__.py",
    "app/api/__init__.py",
    "app/api/endpoints/__init__.py",
    "app/schemas/__init__.py",
    "app/services/__init__.py",
    "app/tasks/__init__.py",
]
for f in init_files:
    open(f, 'a').close()

print("✅ Estrutura de pastas criada!")

---
## 3️⃣ Configuração de Variáveis de Ambiente
`backend/.env`

> ⚠️ Ajuste os valores abaixo para apontar para seus serviços (PostgreSQL, Redis, MinIO).

In [ ]:
# Cria o arquivo .env na raiz
env_content = """POSTGRES_SERVER=127.0.0.1
REDIS_URL=redis://127.0.0.1:6379/0
MINIO_ENDPOINT=127.0.0.1:9000
"""

with open(".env", "w") as f:
    f.write(env_content)

print("✅ Arquivo .env criado!")
print("⚠️  Edite o arquivo .env com suas credenciais reais antes de prosseguir.")

---
## 4️⃣ Core — Configurações
`app/core/config.py`

In [ ]:
%%writefile app/core/config.py
import os
from pathlib import Path
from pydantic_settings import BaseSettings

# Resolve the .env path relative to the backend/ root directory, not the CWD
_BACKEND_DIR = Path(__file__).resolve().parent.parent  # app/core/config.py -> app/ -> backend/
_ENV_FILE = _BACKEND_DIR / ".env"

class Settings(BaseSettings):
    PROJECT_NAME: str = "Termo Fácil"
    API_V1_STR: str = "/api/v1"
    
    # Database Configurations
    POSTGRES_USER: str = "termo_user"
    POSTGRES_PASSWORD: str = "termo_password"
    POSTGRES_SERVER: str = "127.0.0.1"
    POSTGRES_PORT: str = "5432"
    POSTGRES_DB: str = "termo_facil"
    
    # Redis / Celery Configurations
    REDIS_URL: str = "redis://127.0.0.1:6379/0"
    
    # MinIO
    MINIO_ENDPOINT: str = "127.0.0.1:9000"
    MINIO_ACCESS_KEY: str = "admin"
    MINIO_SECRET_KEY: str = "adminpassword"
    MINIO_SECURE: bool = False
    
    @property
    def sync_database_uri(self) -> str:
        return f"postgresql://{self.POSTGRES_USER}:{self.POSTGRES_PASSWORD}@{self.POSTGRES_SERVER}:{self.POSTGRES_PORT}/{self.POSTGRES_DB}"
    
    @property
    def async_database_uri(self) -> str:
        return f"postgresql+asyncpg://{self.POSTGRES_USER}:{self.POSTGRES_PASSWORD}@{self.POSTGRES_SERVER}:{self.POSTGRES_PORT}/{self.POSTGRES_DB}"
        
    class Config:
        case_sensitive = True
        env_file = str(_ENV_FILE)

settings = Settings()

---
## 5️⃣ Core — Celery
`app/core/celery_app.py`

In [ ]:
%%writefile app/core/celery_app.py
from celery import Celery
from app.core.config import settings

celery_app = Celery(
    "termo_facil_worker",
    broker=settings.REDIS_URL,
    backend=settings.REDIS_URL,
)

celery_app.conf.update(
    task_serializer="json",
    accept_content=["json"],
    result_serializer="json",
    timezone="America/Sao_Paulo",
    enable_utc=True,
    # Descobre automaticamente os arquivos de tasks:
    imports=["app.tasks.process_audio"]
)

---
## 6️⃣ Banco de Dados
`app/db.py`

In [ ]:
%%writefile app/db.py
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base
from sqlalchemy.orm import sessionmaker
from app.core.config import settings

# Engine for synchronous connections (useful for simple CRUDs and setup)
engine = create_engine(settings.sync_database_uri, pool_pre_ping=True)

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

---
## 7️⃣ Models (SQLAlchemy)
`app/models.py`

In [ ]:
%%writefile app/models.py
import enum
from sqlalchemy import Column, String, Integer, DateTime, ForeignKey, Text, LargeBinary, JSON, Date, Table, Enum as SQLAlchemyEnum
from sqlalchemy.dialects.postgresql import UUID, JSONB, BYTEA
from sqlalchemy.orm import relationship
import uuid
from datetime import datetime
from app.db import Base

cargo_permissao = Table(
    'cargo_permissao', Base.metadata,
    Column('id_cargo', UUID(as_uuid=True), ForeignKey('cargo.id_cargo'), primary_key=True),
    Column('id_permissao', UUID(as_uuid=True), ForeignKey('permissao.id_permissao'), primary_key=True)
)

# --- ENUMS ---
class CargoUsuario(str, enum.Enum):
    CARGO_1 = 'Cargo 1'
    CARGO_2 = 'Cargo 2'
    DELEGADO = 'Delegado'
    ESCRIVAO = 'Escrivão'
    INVESTIGADOR = 'Investigador'

class TipoDepoente(str, enum.Enum):
    TESTEMUNHA = 'Testemunha'
    VITIMA = 'Vítima'
    SUSPEITO = 'Suspeito'
    INFORMANTE = 'Informante'

class StatusJob(str, enum.Enum):
    PENDENTE = 'Pendente'
    PROCESSANDO = 'Processando'
    CONCLUIDO = 'Concluído'
    ERRO = 'Erro'

class TipoModelo(str, enum.Enum):
    ASR = 'ASR'
    LLM = 'LLM'
    OCR = 'OCR'

# --- MODELS ---
class Delegacia(Base):
    """ Model representing a Police Station (Delegacia) """
    __tablename__ = 'delegacia'
    id_delegacia = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nome_unidade = Column(String(255), nullable=False)
    cod_sinesp = Column(String(100), unique=True, nullable=False)

    usuarios = relationship("Usuario", back_populates="delegacia")
    inqueritos = relationship("Inquerito", back_populates="delegacia")

class Depoente(Base):
    """ Model representing the Testifier/Suspect (Depoente) """
    __tablename__ = 'depoente'
    id_depoente = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    cpf = Column(String(14), unique=True, nullable=False)
    nome_depoente = Column(String(255), nullable=False)

    depoimentos = relationship("Depoimento", back_populates="depoente")

class Modelo(Base):
    """ Model representing an AI Engine version (ASR, LLM, etc.) """
    __tablename__ = 'modelo'
    id_modelo = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nome_modelo = Column(String(255), nullable=False)
    desenvolvedora = Column(String(255), nullable=False)
    tipo_modelo = Column(SQLAlchemyEnum(TipoModelo, name='tipo_modelo_enum', values_callable=lambda obj: [e.value for e in obj]), nullable=False)
    parametros = Column(Text, nullable=True)

class Usuario(Base):
    """ Model representing the Police Officer / Clerk (Escrivão/Delegado) """
    __tablename__ = 'usuario'
    id_usuario = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    id_delegacia = Column(UUID(as_uuid=True), ForeignKey('delegacia.id_delegacia'), nullable=False)
    id_cargo = Column(UUID(as_uuid=True), ForeignKey('cargo.id_cargo'), nullable=False)
    matricula = Column(String(50), unique=True, nullable=False)
    nome = Column(String(255), nullable=False)

    cargo = relationship("Cargo", back_populates="usuarios")
    delegacia = relationship("Delegacia", back_populates="usuarios")
    depoimentos = relationship("Depoimento", back_populates="usuario")

class Inquerito(Base):
    """ Model representing the Police Investigation (Inquérito Policial) """
    __tablename__ = 'inquerito'
    id_inquerito = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    id_delegacia = Column(UUID(as_uuid=True), ForeignKey('delegacia.id_delegacia'), nullable=False)
    num_procedimento = Column(String(100), unique=True, nullable=False)
    data_instauracao = Column(Date, nullable=False)

    delegacia = relationship("Delegacia", back_populates="inqueritos")
    depoimentos = relationship("Depoimento", back_populates="inquerito")

class Depoimento(Base):
    """ Model representing the formal testimony event (Termo de Depoimento) """
    __tablename__ = 'depoimento'
    id_depoimento = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    id_inquerito = Column(UUID(as_uuid=True), ForeignKey('inquerito.id_inquerito'), nullable=False)
    id_usuario = Column(UUID(as_uuid=True), ForeignKey('usuario.id_usuario'), nullable=False)
    id_depoente = Column(UUID(as_uuid=True), ForeignKey('depoente.id_depoente'), nullable=False)
    tipo_depoente = Column(SQLAlchemyEnum(TipoDepoente, name='tipo_depoente_enum', values_callable=lambda obj: [e.value for e in obj]), nullable=False)
    data_hora_reg = Column(DateTime, default=datetime.utcnow)

    inquerito = relationship("Inquerito", back_populates="depoimentos")
    usuario = relationship("Usuario", back_populates="depoimentos")
    depoente = relationship("Depoente", back_populates="depoimentos")
    
    midia_bruta = relationship("MidiaBruta", back_populates="depoimento", uselist=False)
    jobs = relationship("JobProcessamentoIA", back_populates="depoimento")
    termos_finais = relationship("TermosFinais", back_populates="depoimento", uselist=False)

class MidiaBruta(Base):
    """ Model representing the raw audio/video blob metadata and storage path """
    __tablename__ = 'midia_bruta'
    id_depoimento = Column(UUID(as_uuid=True), ForeignKey('depoimento.id_depoimento'), primary_key=True)
    hash_sha256 = Column(String(64), nullable=False)
    storage_path = Column(String(512), nullable=False)
    codec_info = Column(JSONB, nullable=True)

    depoimento = relationship("Depoimento", back_populates="midia_bruta")

class JobProcessamentoIA(Base):
    """ Model tracking the AI background processing queue status for a Testimony """
    __tablename__ = 'job_processamento_ia'
    id_job = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    id_depoimento = Column(UUID(as_uuid=True), ForeignKey('depoimento.id_depoimento'), nullable=False)
    id_modelo_asr = Column(UUID(as_uuid=True), ForeignKey('modelo.id_modelo'), nullable=False)
    id_modelo_llm = Column(UUID(as_uuid=True), ForeignKey('modelo.id_modelo'), nullable=False)
    status = Column(SQLAlchemyEnum(StatusJob, name='status_job_enum', values_callable=lambda obj: [e.value for e in obj]), nullable=False, default=StatusJob.PENDENTE)
    gpu_hw_id = Column(String(100), nullable=True)
    parametros_ia = Column(JSONB, nullable=True)

    depoimento = relationship("Depoimento", back_populates="jobs")
    modelo_asr = relationship("Modelo", foreign_keys=[id_modelo_asr])
    modelo_llm = relationship("Modelo", foreign_keys=[id_modelo_llm])
    
    termos_finais = relationship("TermosFinais", back_populates="job", uselist=False)

class TermosFinais(Base):
    """ Model representing the resulting transcriptions and the final PDF document """
    __tablename__ = 'termos_finais'
    id_depoimento = Column(UUID(as_uuid=True), ForeignKey('depoimento.id_depoimento'), primary_key=True)
    id_job = Column(UUID(as_uuid=True), ForeignKey('job_processamento_ia.id_job'), nullable=False)
    txt_original_ia = Column(Text, nullable=True)
    txt_editado_humano = Column(Text, nullable=True)
    txt_literal_asr = Column(Text, nullable=True)
    dicionario_ner = Column(JSONB, nullable=True)
    assinatura_digital = Column(BYTEA, nullable=True)
    hash_pdf = Column(String(64), nullable=True)
    storage_path_pdf = Column(String(512), nullable=True)

    depoimento = relationship("Depoimento", back_populates="termos_finais")
    job = relationship("JobProcessamentoIA", back_populates="termos_finais")

class Cargo(Base):
    """ Model representing the function of an user """
    __tablename__ = 'cargo'
    id_cargo = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nome_cargo = Column(String(50), nullable=False)

    permissoes = relationship("Permissao", secondary=cargo_permissao, back_populates="cargos")
    usuarios = relationship("Usuario", back_populates="cargo")

class Permissao(Base):
    """ Model representing the permissions of access on the system """
    __tablename__ = 'permissao'
    id_permissao = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nome_permissao = Column(String(50), nullable=False)
    descricao_permissao = Column(Text, nullable=False)

    cargos = relationship("Cargo", secondary=cargo_permissao, back_populates="permissoes")

---
## 8️⃣ Schemas (Pydantic)

### `app/schemas/admin.py`

In [ ]:
%%writefile app/schemas/admin.py
from pydantic import BaseModel, UUID4
from typing import List, Optional

class PermissionSchema(BaseModel):
    id_permissao: UUID4
    nome_permissao: str
    descricao_permissao: str

    class Config:
        from_attributes = True

class CargoSchema(BaseModel):
    id_cargo: UUID4
    nome_cargo: str
    permissoes: List[PermissionSchema] = []

    class Config:
        from_attributes = True

class CargoCreateSchema(BaseModel):
    nome_cargo: str
    permissoes_ids: List[UUID4]

class DelegaciaSchema(BaseModel):
    id_delegacia: UUID4
    nome_unidade: str
    cod_sinesp: str

    class Config:
        from_attributes = True

class UsuarioSchema(BaseModel):
    id_usuario: UUID4
    matricula: str
    nome: str
    id_delegacia: UUID4
    delegacia: Optional[DelegaciaSchema] = None
    cargo: Optional[CargoSchema] = None

    class Config:
        from_attributes = True

class UsuarioUpdateCargoSchema(BaseModel):
    id_cargo: UUID4

### `app/schemas/job.py`

In [ ]:
%%writefile app/schemas/job.py

from pydantic import BaseModel, UUID4
from datetime import datetime
from typing import Optional
from app.models import StatusJob

class JobCreate(BaseModel):
    id_depoimento: UUID4
    # In the MVP, models are mocked, but they could be requested via the payload
    id_modelo_asr: UUID4
    id_modelo_llm: UUID4

class JobResponse(BaseModel):
    id_job: UUID4
    id_depoimento: UUID4
    status: StatusJob
    
    class Config:
        from_attributes = True # Allows Pydantic to read directly from SQLAlchemy models

class JobResultResponse(BaseModel):
    id_job: UUID4
    id_depoimento: UUID4

    txt_original_ia: Optional[str] = None
    txt_editado_humano: Optional[str] = None
    txt_literal_asr: Optional[str] = None

    class Config:
        from_attributes = True

---
## 9️⃣ Services

### `app/services/asr_service.py`

In [ ]:
%%writefile app/services/asr_service.py
import whisper

model = whisper.load_model("base")

def transcrever_audio(audio_path: str) -> str:
    """Transcreve o áudio para texto"""
    result = model.transcribe(audio_path, language="en")
    return result["text"]

# ====================================================
#                       Testes
# ====================================================

def main():
    """Função principal para testar a transcrição"""
    audio_path = "../sample_audio/micro-machines.wav"
    texto = transcrever_audio(audio_path)
    print(texto)

if __name__ == "__main__":
    main()

### `app/services/minio_service.py`

In [ ]:
%%writefile app/services/minio_service.py
import boto3
from botocore.client import Config
from app.core.config import settings

class MinioService:
    def __init__(self):
        # MinIO API is 100% compatible with AWS S3
        self.s3_client = boto3.client(
            's3',
            endpoint_url=f"http://{settings.MINIO_ENDPOINT}",
            aws_access_key_id=settings.MINIO_ACCESS_KEY,
            aws_secret_access_key=settings.MINIO_SECRET_KEY,
            config=Config(signature_version='s3v4'),
            region_name='us-east-1' # Fictional region required by boto3
        )
        self.bucket_name = "audio-uploads"
        self._ensure_bucket_exists(self.bucket_name)

    def _ensure_bucket_exists(self, bucket_name: str):
        try:
            self.s3_client.head_bucket(Bucket=bucket_name)
        except Exception:
            # Bucket does not exist, so we create it
            self.s3_client.create_bucket(Bucket=bucket_name)

    def upload_file(self, file_content: bytes, file_name: str, bucket_name: str = "audio-uploads") -> str:
        """
        Uploads the file and returns its storage URI.
        """
        self._ensure_bucket_exists(bucket_name)
        self.s3_client.put_object(
            Bucket=bucket_name,
            Key=file_name,
            Body=file_content
        )
        # The storage_path to be saved in the database
        return f"s3://{bucket_name}/{file_name}"
        
    def generate_presigned_url(self, bucket_name: str, object_name: str, expiration: int = 3600) -> str:
        """
        Generate a presigned URL to share an S3 object
        """
        response = self.s3_client.generate_presigned_url('get_object',
                                                    Params={'Bucket': bucket_name,
                                                            'Key': object_name},
                                                    ExpiresIn=expiration)
        return response

minio_service = MinioService()

### `app/services/pdf_service.py`

In [ ]:
%%writefile app/services/pdf_service.py
import io
from fastapi import HTTPException
from sqlalchemy.orm import Session
from app.models import Depoimento
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

def gerar_pdf_termo_depoimento(db: Session, id_depoimento) -> bytes:
    """
    Fetches the complete testimony data from the database and generates 
    the official SSP-PI PDF document using ReportLab's Platypus engine.
    """
    # 1. Fetch the testimony record along with its database relationships
    depoimento = db.query(Depoimento).filter(Depoimento.id_depoimento == id_depoimento).first()
    if not depoimento:
        raise HTTPException(status_code=404, detail="Depoimento não encontrado.")
    
    termos_finais = depoimento.termos_finais
    if not termos_finais:
        raise HTTPException(status_code=404, detail="Termos de transcrição e síntese da IA não encontrados para este depoimento.")
        
    # Prioritize human-edited text over raw AI synthesis
    texto_final = termos_finais.txt_editado_humano or termos_finais.txt_original_ia
    if not texto_final:
        raise HTTPException(status_code=400, detail="Não há texto de depoimento disponível para gerar o documento.")

    # 2. Setup memory buffer and document template with strict margins
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(
        buffer, 
        pagesize=letter, 
        rightMargin=54, 
        leftMargin=54, 
        topMargin=54, 
        bottomMargin=54
    )
    story = []
    
    # 3. Initialize and configure custom paragraph styles
    styles = getSampleStyleSheet()
    
    header_style = ParagraphStyle(
        'SSP_Header',
        fontName='Helvetica-Bold',
        fontSize=10,
        leading=14,
        alignment=1 # Center alignment
    )
    
    title_style = ParagraphStyle(
        'SSP_Title',
        fontName='Helvetica-Bold',
        fontSize=13,
        leading=18,
        alignment=1,
        spaceAfter=15,
        spaceBefore=10
    )
    
    label_bold_style = ParagraphStyle(
        'SSP_Label',
        fontName='Helvetica-Bold',
        fontSize=10,
        leading=14
    )
    
    body_style = ParagraphStyle(
        'SSP_Body',
        fontName='Helvetica',
        fontSize=11,
        leading=16,
        alignment=4, # Justified alignment
        spaceAfter=12
    )
    
    center_text_style = ParagraphStyle(
        'SSP_CenterSign', 
        fontName='Helvetica', 
        fontSize=10, 
        leading=14,
        alignment=1
    )
    
    italic_center_style = ParagraphStyle(
        'SSP_ItalicSign', 
        fontName='Helvetica-Oblique', 
        fontSize=9, 
        leading=12,
        alignment=1
    )

    # 4. Build the official Header Section
    story.append(Paragraph("ESTADO DO PIAUÍ", header_style))
    story.append(Paragraph("SECRETARIA DE SEGURANÇA PÚBLICA", header_style))
    nome_delegacia = depoimento.inquerito.delegacia.nome_unidade.upper() if depoimento.inquerito and depoimento.inquerito.delegacia else "DELEGACIA DE POLÍCIA CIVIL"
    story.append(Paragraph(nome_delegacia, header_style))
    story.append(Spacer(1, 15))
    
    # 5. Build the Document Title
    tipo_depoente_str = depoimento.tipo_depoente.value.upper() if depoimento.tipo_depoente else "DEPOENTE"
    story.append(Paragraph(f"TERMO DE DEPOIMENTO ({tipo_depoente_str})", title_style))
    
    # 6. Extract metadata info safely
    num_proc = depoimento.inquerito.num_procedimento if depoimento.inquerito else "Não informado"
    data_inst = depoimento.inquerito.data_instauracao.strftime('%d/%m/%Y') if depoimento.inquerito and depoimento.inquerito.data_instauracao else "--/--/----"
    nome_usuario = depoimento.usuario.nome if depoimento.usuario else "Não informado"
    matricula_usuario = depoimento.usuario.matricula if depoimento.usuario else "N/A"
    
    nome_depoente = depoimento.depoente.nome_depoente if depoimento.depoente else "Não informado"
    cpf_depoente = depoimento.depoente.cpf if depoimento.depoente else "Não informado"
    condicao = depoimento.tipo_depoente.value if depoimento.tipo_depoente else "Não especificado"
    
    # 7. Assemble the Case Metadata Table
    meta_data = [
        [Paragraph("Procedimento / IP nº:", label_bold_style), Paragraph(num_proc, styles['Normal'])],
        [Paragraph("Data de Instauração:", label_bold_style), Paragraph(data_inst, styles['Normal'])],
        [Paragraph("Autoridade Responsável:", label_bold_style), Paragraph(f"{nome_usuario} (Matrícula: {matricula_usuario})", styles['Normal'])],
        [Paragraph("Nome do Depoente:", label_bold_style), Paragraph(nome_depoente, styles['Normal'])],
        [Paragraph("CPF:", label_bold_style), Paragraph(cpf_depoente, styles['Normal'])],
        [Paragraph("Condição Jurídica:", label_bold_style), Paragraph(condicao, styles['Normal'])],
    ]
    
    # Set explicit column widths to guarantee structural alignment
    meta_table = Table(meta_data, colWidths=[140, 364])
    meta_table.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ('BOTTOMPADDING', (0,0), (-1,-1), 3),
        ('TOPPADDING', (0,0), (-1,-1), 3),
        ('LINEBELOW', (0,5), (1,5), 1, colors.gray), # Divider line after metadata
    ]))
    
    story.append(meta_table)
    story.append(Spacer(1, 15))
    
    # 8. Build the Statement Content Section
    story.append(Paragraph("DEPOIMENTO / DECLARAÇÕES PRESTADAS", label_bold_style))
    story.append(Spacer(1, 8))
    
    # Replace newlines with HTML line breaks to ensure correct text rendering in Paragraph flows
    formatted_text = texto_final.replace('\n', '<br/>')
    story.append(Paragraph(formatted_text, body_style))
    story.append(Spacer(1, 20))
    
    # 9. Build Closing Statement and Signature Fields
    story.append(Paragraph("Nada mais havendo a declarar, foi lavrado o presente termo.", styles['Normal']))
    story.append(Spacer(1, 45))
    
    # Authority Signature Block
    story.append(Paragraph("__________________________________________________", center_text_style))
    story.append(Paragraph(nome_usuario, center_text_style))
    story.append(Paragraph("Autoridade Policial / Escrivão de Polícia", italic_center_style))
    
    story.append(Spacer(1, 40))
    
    # Testifier Signature Block
    story.append(Paragraph("__________________________________________________", center_text_style))
    story.append(Paragraph(nome_depoente, center_text_style))
    story.append(Paragraph(f"Depoente ({condicao})", italic_center_style))
    
    # 10. Compile the document flow and extract content bytes
    doc.build(story)
    pdf_bytes = buffer.getvalue()
    buffer.close()
    
    import hashlib
    sha256 = hashlib.sha256(pdf_bytes).hexdigest()
    
    return pdf_bytes, sha256

---
## 🔟 Tasks (Celery)
`app/tasks/process_audio.py`

In [ ]:
%%writefile app/tasks/process_audio.py
import time
import logging
from app.core.celery_app import celery_app
from app.db import SessionLocal
from app.models import JobProcessamentoIA, StatusJob, TermosFinais
import hashlib

logger = logging.getLogger(__name__)

mock_asr = (
    "[00:02] INQUIRIDOR: Qual o seu nome completo e onde você estava ontem às 22h?\n"
    "[00:08] DEPOENTE: Meu nome é Carlos Eduardo Alves. Eu estava na praça central "
    "quando vi dois homens numa moto preta passarem muito rápido. A placa parecia ser ABC-1234.\n"
    "[00:15] INQUIRIDOR: Você notou alguma característica física ou armada?\n"
    "[00:21] DEPOENTE: O garupa tava com uma jaqueta vermelha e parecia segurar uma arma preta."
)
mock_llm = (
    "Aos costumes, disse chamar-se Carlos Eduardo Alves. Inquirido pela autoridade policial "
    "acerca dos fatos ocorridos na data de ontem, por volta das 22h00min, respondeu que se "
    "encontrava na praça central desta urbe, momento em que avistou dois indivíduos trafegando "
    "em alta velocidade em uma motocicleta de cor preta, cuja placa ostentava o alfanumérico "
    "parcial ABC-1234. Relatou ainda que o indivíduo que ocupava a garupa trajava jaqueta "
    "vermelha e aparentemente portava arma de fogo de cor preta."
)
mock_ner = {
    "PESSOAS": ["Carlos Eduardo Alves"],
    "LOCAIS": ["Praça central"],
    "VEICULOS": ["Moto preta", "Placa ABC-1234"],
    "OBJETOS_CRIME": ["Jaqueta vermelha", "Arma preta"]
}

@celery_app.task(name="process_audio")
def process_audio_task(job_id: str):
    """
    Simulates the Processing Pipeline (ASR -> NER -> LLM).
    In production (HPC Mandu), this would call the local GPUs.
    """
    logger.info(f"Iniciando processamento do Job ID: {job_id}")
    
    db = SessionLocal()
    try:
        # 1. Fetch the Job record
        job = db.query(JobProcessamentoIA).filter(JobProcessamentoIA.id_job == job_id).first()
        if not job:
            logger.error(f"Job {job_id} not found in database.")
            return False
            
        # 2. Update Status to Processando (Processing)
        job.status = StatusJob.PROCESSANDO
        db.commit()
        
        # 3. Simulate ASR inference (Whisper)
        logger.info("Running ASR (Whisper)...")
        time.sleep(3) # Simulates VRAM processing time
        
        # 4. Simulate Fact Anchoring (LeNER-Br)
        logger.info("Extracting Entities (LeNER-Br)...")
        time.sleep(2)
        
        # 5. Simulate Legal Synthesis (vLLM Temperature 0.0)
        logger.info("Generating Deterministic Legal Summary (LLM)...")
        time.sleep(4)
        
        resultado_final = TermosFinais(
            id_depoimento=job.id_depoimento,
            id_job=job.id_job,
            txt_literal_asr=mock_asr,
            txt_original_ia=mock_llm,
            dicionario_ner=mock_ner,
            txt_editado_humano=None,
            assinatura_digital=None,
            hash_pdf=None
        )

        db.add(resultado_final)

        # 6. Mark as successful
        job.status = StatusJob.CONCLUIDO
        db.commit()
        logger.info(f"Job {job_id} finalizado com sucesso!")
        
        return True

    except Exception as e:
        logger.error(f"Error processing Job {job_id}: {str(e)}")
        # We could set StatusJob.ERRO here
        if job:
            job.status = StatusJob.ERRO
            db.commit()
        return False
        
    finally:
        db.close()

---
## 1️⃣1️⃣ API — Dependencies
`app/api/deps.py`

In [ ]:
%%writefile app/api/deps.py
from fastapi import Depends, HTTPException, status, Header
from sqlalchemy.orm import Session
from app.db import get_db
from app.models import Usuario
import uuid

def get_current_user(
    db: Session = Depends(get_db),
    x_user_id: str = Header(None, alias="X-User-Id")
) -> Usuario:
    """
    Dependency to retrieve the currently logged in / simulated user.
    Reads from the 'X-User-Id' header. If absent, falls back to the first user in the database.
    """
    if x_user_id:
        try:
            user_uuid = uuid.UUID(x_user_id)
            user = db.query(Usuario).filter(Usuario.id_usuario == user_uuid).first()
            if not user:
                raise HTTPException(
                    status_code=status.HTTP_401_UNAUTHORIZED,
                    detail="Usuário não encontrado no sistema."
                )
            return user
        except ValueError:
            raise HTTPException(
                status_code=status.HTTP_400_BAD_REQUEST,
                detail="Formato de ID de usuário inválido."
            )
    else:
        # Fallback for dev / mock environment
        user = db.query(Usuario).first()
        if not user:
            raise HTTPException(
                status_code=status.HTTP_401_UNAUTHORIZED,
                detail="Nenhum usuário cadastrado no banco de dados. Por favor, execute o seed."
            )
        return user

class RequirePermission:
    """
    Dependency to require a specific permission for an API endpoint.
    Example: Depends(RequirePermission('GERAR_PDF'))
    """
    def __init__(self, permission_name: str):
        self.permission_name = permission_name

    def __call__(
        self,
        current_user: Usuario = Depends(get_current_user),
        db: Session = Depends(get_db)
    ) -> Usuario:
        if not current_user.cargo:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail="O usuário não possui nenhum cargo atribuído."
            )
        
        # Check permissions associated with the user's role (cargo)
        permissions = [p.nome_permissao for p in current_user.cargo.permissoes]
        if self.permission_name not in permissions:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Acesso negado: Permissão '{self.permission_name}' é necessária."
            )
        
        return current_user

---
## 1️⃣2️⃣ API — Endpoints

### `app/api/endpoints/auth.py`

In [ ]:
%%writefile app/api/endpoints/auth.py
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from typing import List
from app.db import get_db
from app.models import Usuario
from app.schemas.admin import UsuarioSchema
from app.api.deps import get_current_user

router = APIRouter()

@router.get("/me", response_model=UsuarioSchema)
def get_me(current_user: Usuario = Depends(get_current_user)):
    """
    Get the currently active simulated user.
    """
    return current_user

@router.get("/users", response_model=List[UsuarioSchema])
def list_sim_users(db: Session = Depends(get_db)):
    """
    List all users in the system (no protection, used for user switching in frontend).
    """
    return db.query(Usuario).all()

### `app/api/endpoints/admin.py`

In [ ]:
%%writefile app/api/endpoints/admin.py
from fastapi import APIRouter, Depends, HTTPException, status
from sqlalchemy.orm import Session
from typing import List
import uuid
from app.db import get_db
from app.models import Usuario, Cargo, Permissao
from app.schemas.admin import (
    UsuarioSchema, UsuarioUpdateCargoSchema,
    CargoSchema, CargoCreateSchema, PermissionSchema
)
from app.api.deps import RequirePermission

# Require 'GERENCIAR_USUARIOS' permission for all routes in this controller
router = APIRouter(dependencies=[Depends(RequirePermission('GERENCIAR_USUARIOS'))])

@router.get("/users", response_model=List[UsuarioSchema])
def list_users(db: Session = Depends(get_db)):
    """
    List all users in the system.
    """
    return db.query(Usuario).all()

@router.put("/users/{user_id}/cargo", response_model=UsuarioSchema)
def update_user_cargo(user_id: uuid.UUID, payload: UsuarioUpdateCargoSchema, db: Session = Depends(get_db)):
    """
    Update the cargo (role) of a user.
    """
    user = db.query(Usuario).filter(Usuario.id_usuario == user_id).first()
    if not user:
        raise HTTPException(status_code=404, detail="Usuário não encontrado.")
        
    cargo = db.query(Cargo).filter(Cargo.id_cargo == payload.id_cargo).first()
    if not cargo:
        raise HTTPException(status_code=404, detail="Cargo não encontrado.")
        
    user.id_cargo = payload.id_cargo
    db.commit()
    db.refresh(user)
    return user

@router.get("/cargos", response_model=List[CargoSchema])
def list_cargos(db: Session = Depends(get_db)):
    """
    List all roles (cargos) and their permissions.
    """
    return db.query(Cargo).all()

@router.post("/cargos", response_model=CargoSchema, status_code=201)
def create_cargo(payload: CargoCreateSchema, db: Session = Depends(get_db)):
    """
    Create a new role (cargo) and associate selected permissions.
    """
    existing_cargo = db.query(Cargo).filter(Cargo.nome_cargo == payload.nome_cargo).first()
    if existing_cargo:
        raise HTTPException(status_code=400, detail="Já existe um cargo com este nome.")
        
    # Retrieve permissions from DB
    permissions = db.query(Permissao).filter(Permissao.id_permissao.in_(payload.permissoes_ids)).all()
    if len(permissions) != len(payload.permissoes_ids):
        raise HTTPException(status_code=400, detail="Alguma das permissões informadas não foi encontrada.")
        
    new_cargo = Cargo(
        nome_cargo=payload.nome_cargo,
        permissoes=permissions
    )
    db.add(new_cargo)
    db.commit()
    db.refresh(new_cargo)
    return new_cargo

@router.get("/permissions", response_model=List[PermissionSchema])
def list_permissions(db: Session = Depends(get_db)):
    """
    List all available permissions in the system.
    """
    return db.query(Permissao).all()

### `app/api/endpoints/jobs.py`

In [ ]:
%%writefile app/api/endpoints/jobs.py
from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.orm import Session
from app.db import get_db
from app.models import JobProcessamentoIA, TermosFinais
from app.schemas.job import JobResponse, JobResultResponse
from app.api.deps import RequirePermission
import uuid

router = APIRouter(dependencies=[Depends(RequirePermission('EDITAR_TERMO'))])

@router.get("/{job_id}", response_model=JobResponse)
def get_job_status(job_id: uuid.UUID, db: Session = Depends(get_db)):
    """
    Returns the current status of a processing Job in the queue.
    """
    job = db.query(JobProcessamentoIA).filter(JobProcessamentoIA.id_job == job_id).first()
    
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
        
    return job

@router.get("/{job_id}/resultado", response_model=JobResultResponse)
def get_job_result(job_id: uuid.UUID, db: Session = Depends(get_db)):
    """
    Returns the result of a Job processing
    """

    termos_finais = db.query(TermosFinais).filter(TermosFinais.id_job == job_id).first()

    if not termos_finais:
        raise HTTPException(status_code=404, detail="No transcription was found for this Job")
    
    return termos_finais

### `app/api/endpoints/upload.py`

In [ ]:
%%writefile app/api/endpoints/upload.py
from fastapi import APIRouter, Depends, UploadFile, File, Form, HTTPException
from sqlalchemy.orm import Session
from app.db import get_db
from app.services.minio_service import minio_service
from app.models import MidiaBruta, JobProcessamentoIA
from app.schemas.job import JobResponse
from app.api.deps import RequirePermission
import uuid
import hashlib

router = APIRouter()

@router.post("/audio", response_model=JobResponse, status_code=202)
async def upload_audio(
    id_depoimento: str = Form(...),
    id_modelo_asr: str = Form(...),
    id_modelo_llm: str = Form(...),
    file: UploadFile = File(...),
    db: Session = Depends(get_db),
    current_user = Depends(RequirePermission('UPLOAD_AUDIO'))
):
    """
    Receives an audio file, saves it in MinIO and creates the initial Job in PostgreSQL.
    """
    # 1. Basic extension validation
    if not file.filename.endswith(('.wav', '.mp3', '.m4a')):
        raise HTTPException(status_code=400, detail="Unsupported audio format.")
        
    content = await file.read()
    
    # 2. Upload to MinIO
    unique_filename = f"{uuid.uuid4()}_{file.filename}"
    storage_path = minio_service.upload_file(content, unique_filename)
    
    # File hash for integrity (SHA256)
    file_hash = hashlib.sha256(content).hexdigest()
    
    # 3. Save or update raw media record to Database
    media_record = db.query(MidiaBruta).filter(MidiaBruta.id_depoimento == id_depoimento).first()
    if media_record:
        media_record.hash_sha256 = file_hash
        media_record.storage_path = storage_path
        media_record.codec_info = {"filename": file.filename, "content_type": file.content_type}
    else:
        media_record = MidiaBruta(
            id_depoimento=id_depoimento,
            hash_sha256=file_hash,
            storage_path=storage_path,
            codec_info={"filename": file.filename, "content_type": file.content_type}
        )
        db.add(media_record)
    
    # 4. Create the Job record
    job_record = JobProcessamentoIA(
        id_depoimento=id_depoimento,
        id_modelo_asr=id_modelo_asr,
        id_modelo_llm=id_modelo_llm
        # Status defaults to PENDENTE
    )
    db.add(job_record)
    db.commit()
    db.refresh(job_record)
    
    # Trigger the background Celery task
    from app.core.celery_app import celery_app
    celery_app.send_task("process_audio", args=[str(job_record.id_job)])
    
    return job_record

### `app/api/endpoints/pdf.py`

In [ ]:
%%writefile app/api/endpoints/pdf.py
# backend/app/api/endpoints/pdf.py
from fastapi import APIRouter, Depends, HTTPException, Response
from sqlalchemy.orm import Session
from pydantic import BaseModel, UUID4
from app.db import get_db
from app.models import TermosFinais
from app.api.deps import RequirePermission
import hashlib
import uuid

# Importing the PDF generation service function
from app.services.pdf_service import gerar_pdf_termo_depoimento

router = APIRouter()

class PDFGeneratePayload(BaseModel):
    id_depoimento: UUID4

# Importing minio service
from app.services.minio_service import minio_service

@router.post("/gerar", dependencies=[Depends(RequirePermission('GERAR_PDF'))])
def gerar_pdf(payload: PDFGeneratePayload, db: Session = Depends(get_db)):
    """
    Generate the official testimony PDF, upload to MinIO, and return a presigned URL.
    """
    termos = db.query(TermosFinais).filter(TermosFinais.id_depoimento == payload.id_depoimento).first()
    if not termos:
        raise HTTPException(status_code=404, detail="Termos finais do depoimento não encontrados.")
    
    # Generate PDF bytes and its SHA256 hash
    try:
        pdf_bytes, sha256_hash = gerar_pdf_termo_depoimento(db, payload.id_depoimento)
    except HTTPException as http_exc:
        raise http_exc
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro interno ao gerar o arquivo PDF: {str(e)}")

    # Upload to MinIO
    bucket_name = "termos-finais"
    object_name = f"{payload.id_depoimento}/termo.pdf"
    
    try:
        storage_path = minio_service.upload_file(pdf_bytes, object_name, bucket_name)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro ao fazer upload do PDF para o MinIO: {str(e)}")
        
    # Generate presigned URL
    try:
        presigned_url = minio_service.generate_presigned_url(bucket_name, object_name, expiration=3600)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro ao gerar URL de download: {str(e)}")
    
    # Save metadata to DB
    termos.hash_pdf = sha256_hash
    termos.storage_path_pdf = storage_path
    db.commit()
    
    return {
        "status": "success",
        "message": "PDF gerado e assinado digitalmente com sucesso!",
        "id_depoimento": str(payload.id_depoimento),
        "hash_pdf": sha256_hash,
        "pdf_url": presigned_url
    }

@router.get("/{job_id}/pdf")
def download_job_pdf(job_id: uuid.UUID, db: Session = Depends(get_db)):
    """
    Generates and triggers the download of the official PDF document 
    for the completed testimony related to the given Job ID.
    """
    termos_finais = db.query(TermosFinais).filter(TermosFinais.id_job == job_id).first()
    if not termos_finais:
        raise HTTPException(status_code=404, detail="Resultado do processamento não encontrado para este Job.")
    
    try:
        pdf_content = gerar_pdf_termo_depoimento(db, termos_finais.id_depoimento)
    except HTTPException as http_exc:
        raise http_exc
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro interno ao gerar o arquivo PDF: {str(e)}")
        
    filename = f"termo_depoimento_{termos_finais.id_depoimento}.pdf"
    return Response(
        content=pdf_content,
        media_type="application/pdf",
        headers={"Content-Disposition": f"attachment; filename={filename}"}
    )

---
## 1️⃣3️⃣ API — Router Principal
`app/api/api.py`

In [ ]:
%%writefile app/api/api.py
from fastapi import APIRouter
from app.api.endpoints import upload, jobs, admin, auth, pdf

api_router = APIRouter()

api_router.include_router(upload.router, prefix="/upload", tags=["Upload & Ingestion"])
api_router.include_router(jobs.router, prefix="/jobs", tags=["Jobs Queue"])
api_router.include_router(admin.router, prefix="/admin", tags=["Admin & RBAC"])
api_router.include_router(auth.router, prefix="/auth", tags=["Auth Simulator"])
api_router.include_router(pdf.router, prefix="/pdf", tags=["PDF Generation"])

---
## 1️⃣4️⃣ Aplicação Principal
`app/main.py`

In [ ]:
%%writefile app/main.py
import sys
import os
# Hack to allow running main.py directly from the IDE ("Run" button) from the root folder
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from app.core.config import settings

app = FastAPI(
    title=settings.PROJECT_NAME,
    description="Sistema Inteligente de Redação de Termos de Depoimentos - SSP-PI",
    version="1.0.0",
    openapi_url=f"{settings.API_V1_STR}/openapi.json"
)

# CORS Configuration (Cross-Origin Resource Sharing)
# In production, origins should be restricted to the SSP-PI network
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health", tags=["System"])
async def health_check():
    """
    Endpoint for API health check.
    """
    return {"status": "ok", "system": settings.PROJECT_NAME}

from app.api.api import api_router

# Include API routes
app.include_router(api_router, prefix=settings.API_V1_STR)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

---
## 1️⃣5️⃣ Script de Seed do Banco de Dados
`scripts/seed_db.py`

In [ ]:
%%writefile scripts/seed_db.py
import sys
import os
from datetime import date
import uuid
import json

# Adiciona a raiz do projeto ao path
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from app.db import SessionLocal, engine, Base
from app.models import (
    Delegacia, Usuario, Depoente, Inquerito, Depoimento, Modelo,
    TipoDepoente, TipoModelo, Cargo, Permissao
)

def seed():
    # Garante que as novas tabelas (Cargo, Permissao) existam no banco!
    Base.metadata.create_all(bind=engine)
    
    db = SessionLocal()
    try:
        # Verifica se já existe um modelo ASR
        modelo_asr = db.query(Modelo).filter(Modelo.tipo_modelo == TipoModelo.ASR).first()
        if not modelo_asr:
            modelo_asr = Modelo(
                nome_modelo="Whisper Turbo (Mock)",
                desenvolvedora="OpenAI",
                tipo_modelo=TipoModelo.ASR
            )
            db.add(modelo_asr)

        # Verifica se já existe um modelo LLM
        modelo_llm = db.query(Modelo).filter(Modelo.tipo_modelo == TipoModelo.LLM).first()
        if not modelo_llm:
            modelo_llm = Modelo(
                nome_modelo="vLLM Llama 3 (Mock)",
                desenvolvedora="Meta",
                tipo_modelo=TipoModelo.LLM
            )
            db.add(modelo_llm)

        # Delegacia
        delegacia = db.query(Delegacia).first()
        if not delegacia:
            delegacia = Delegacia(
                nome_unidade="12ª Delegacia de Polícia",
                cod_sinesp="12DP-PI"
            )
            db.add(delegacia)
            db.flush() # Para gerar o ID
            
        # Permissões
        permissoes_chaves = ['UPLOAD_AUDIO', 'EDITAR_TERMO', 'GERAR_PDF', 'GERENCIAR_USUARIOS']
        permissoes_obj = {}
        for p in permissoes_chaves:
            perm = db.query(Permissao).filter(Permissao.nome_permissao == p).first()
            if not perm:
                perm = Permissao(nome_permissao=p, descricao_permissao=f"Permite {p}")
                db.add(perm)
                db.flush() # Flush to get ID if needed inside the loop, though we get the object reference
            else:
                permissoes_obj[p] = perm
                
        # If we added new ones, we need them in the dictionary
        for p in permissoes_chaves:
            if p not in permissoes_obj:
                permissoes_obj[p] = db.query(Permissao).filter(Permissao.nome_permissao == p).first()

        # Cargos
        cargo_admin = db.query(Cargo).filter(Cargo.nome_cargo == 'Admin').first()
        if not cargo_admin:
            cargo_admin = Cargo(nome_cargo='Admin')
            cargo_admin.permissoes = list(permissoes_obj.values())
            db.add(cargo_admin)
            
        cargo_escrivao = db.query(Cargo).filter(Cargo.nome_cargo == 'Escrivão').first()
        if not cargo_escrivao:
            cargo_escrivao = Cargo(nome_cargo='Escrivão')
            cargo_escrivao.permissoes = [permissoes_obj['UPLOAD_AUDIO'], permissoes_obj['EDITAR_TERMO'], permissoes_obj['GERAR_PDF']]
            db.add(cargo_escrivao)

        cargo_delegado = db.query(Cargo).filter(Cargo.nome_cargo == 'Delegado').first()
        if not cargo_delegado:
            cargo_delegado = Cargo(nome_cargo='Delegado')
            cargo_delegado.permissoes = [permissoes_obj['EDITAR_TERMO'], permissoes_obj['GERAR_PDF']]
            db.add(cargo_delegado)
        
        db.flush()

        # Usuarios
        usuario_escrivao = db.query(Usuario).filter(Usuario.matricula == "123456").first()
        if not usuario_escrivao:
            usuario_escrivao = Usuario(
                id_delegacia=delegacia.id_delegacia,
                id_cargo=cargo_escrivao.id_cargo,
                matricula="123456",
                nome="João Silva (Escrivão)"
            )
            db.add(usuario_escrivao)

        usuario_delegado = db.query(Usuario).filter(Usuario.matricula == "789012").first()
        if not usuario_delegado:
            usuario_delegado = Usuario(
                id_delegacia=delegacia.id_delegacia,
                id_cargo=cargo_delegado.id_cargo,
                matricula="789012",
                nome="Maria Souza (Delegado)"
            )
            db.add(usuario_delegado)

        usuario_admin = db.query(Usuario).filter(Usuario.matricula == "111111").first()
        if not usuario_admin:
            usuario_admin = Usuario(
                id_delegacia=delegacia.id_delegacia,
                id_cargo=cargo_admin.id_cargo,
                matricula="111111",
                nome="Carlos Admin (Admin)"
            )
            db.add(usuario_admin)

        # Inquerito
        inquerito = db.query(Inquerito).first()
        if not inquerito:
            inquerito = Inquerito(
                id_delegacia=delegacia.id_delegacia,
                num_procedimento="IP-2026/001",
                data_instauracao=date.today()
            )
            db.add(inquerito)
            
        # Depoente
        depoente = db.query(Depoente).first()
        if not depoente:
            depoente = Depoente(
                cpf="00011122233",
                nome_depoente="José Maria da Silva"
            )
            db.add(depoente)
            db.flush()

        db.commit()  # Persiste cargos, usuários, inquerito e depoente

        # Depoimento
        depoimento = db.query(Depoimento).first()
        if not depoimento:
            depoimento = Depoimento(
                id_inquerito=inquerito.id_inquerito,
                id_usuario=usuario_escrivao.id_usuario,
                id_depoente=depoente.id_depoente,
                tipo_depoente=TipoDepoente.SUSPEITO
            )
            db.add(depoimento)
            db.commit()

        # Agora recupera os IDs finais
        db.refresh(depoimento)
        db.refresh(modelo_asr)
        db.refresh(modelo_llm)

        print("=== Banco de dados populado com sucesso! ===")
        print(f"ID Depoimento: {depoimento.id_depoimento}")
        print(f"ID Modelo ASR: {modelo_asr.id_modelo}")
        print(f"ID Modelo LLM: {modelo_llm.id_modelo}")
        print("Use estes IDs para testar o upload!")
        
        # Salva num arquivo json na raiz do frontend para podermos usar os IDs automaticamente no Mock
        mock_data = {
            "id_depoimento": str(depoimento.id_depoimento),
            "id_modelo_asr": str(modelo_asr.id_modelo),
            "id_modelo_llm": str(modelo_llm.id_modelo)
        }
        
        frontend_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../frontend/src/mock_ids.json"))
        with open(frontend_path, "w") as f:
            json.dump(mock_data, f)
            
        print(f"IDs salvos em: {frontend_path}")

    except Exception as e:
        print(f"Erro ao rodar seed: {e}")
        db.rollback()
    finally:
        db.close()

if __name__ == "__main__":
    seed()

---
## 1️⃣6️⃣ Executar o Seed do Banco de Dados

> ⚠️ Certifique-se de que o PostgreSQL está rodando e as configurações no `.env` estão corretas antes de executar esta célula.

In [ ]:
import sys
sys.path.insert(0, '.')

from scripts.seed_db import seed
seed()

---
## 1️⃣7️⃣ Iniciar o Servidor FastAPI

> 💡 No Colab, use `nest_asyncio` para rodar o servidor uvicorn dentro do notebook. Após rodar esta célula, o servidor ficará disponível na porta **8000**.  
> Para expô-lo publicamente, use `ngrok` ou o túnel nativo do Colab.

In [ ]:
# Instala nest_asyncio para permitir rodar uvicorn dentro do Colab
!pip install -q nest_asyncio

import nest_asyncio
nest_asyncio.apply()

import uvicorn
import sys
sys.path.insert(0, '.')

from app.main import app

print("🚀 Iniciando servidor Termo Fácil na porta 8000...")
print("📚 Acesse a documentação em: http://localhost:8000/api/v1/openapi.json")

uvicorn.run(app, host="0.0.0.0", port=8000)

---
## 1️⃣8️⃣ (Opcional) Expor com ngrok

Se quiser acessar a API de fora do Colab, use ngrok para criar um túnel público.

In [ ]:
# Instala pyngrok
!pip install -q pyngrok

from pyngrok import ngrok

# Cole seu authtoken do ngrok aqui: https://dashboard.ngrok.com/
# ngrok.set_auth_token("SEU_TOKEN_AQUI")

public_url = ngrok.connect(8000)
print(f"🌐 API disponível publicamente em: {public_url}")
print(f"📚 Swagger UI: {public_url}/api/v1/openapi.json")